# Notebook 16 (Revised) — SOLETE **Historically Exposed Retrospective Evaluation**

**Status:** Completed — SOLETE historically exposed retrospective evaluation executed and verified. Three frozen LightGBM models were scored against two predetermined persistence baselines. No refitting, hyperparameter tuning, or model-selection changes were performed.

This notebook follows the finalized **read-only Notebook 15 evaluation-protocol preflight**, committed at `02e7526`.

**Research sequence:** Complete the SOLETE retrospective evaluation first, then proceed to PVOD external validation.

---

## Scientific Classification and Interpretation

**Important:** The June–August 2019 SOLETE evaluation period was previously examined in the historical research lineage.

Accordingly, all results in this notebook must be described as:

**Historically exposed retrospective evaluation results — not an untouched holdout, fresh independent test, or independent confirmation.**

The revised workflow excluded these observations from development cross-validation, model selection, and final-model refitting. However, that separation does not erase their earlier exposure in the broader project.

Historical terminal results must not be compared directly with revised development-CV estimates as though they were generated under identical evaluation and selection procedures.

No model switching, additional tuning, or retrospective selection is permitted after observing these results.

## Frozen Development-Trained Models

The following model configurations were finalized before the retrospective evaluation.

| Horizon | Frozen model | Predictor family | Model artifact | Development training rows |
|---:|---|---|---|---:|
| 15 min | LightGBM C1 | F1 — 133 features | `lightgbm_c1_h15_F1.txt` | 104,817 |
| 30 min | LightGBM C1 | F2 — 138 features | `lightgbm_c1_h30_F2.txt` | 104,814 |
| 60 min | LightGBM C1 | F2 — 138 features | `lightgbm_c1_h60_F2.txt` | 104,808 |

The frozen configurations use:

- **Learning rate:** 0.03
- **Boosting iterations:** 300
- **Number of leaves:** 15
- **Random seed:** 42
- **Early stopping:** Disabled

The model-selection lock was committed at `c6a9162`. The three fitted-model provenance records and independent verification script were committed at `565d431`.

## Predeclared Evaluation Protocol

The evaluation methodology was established in Notebook 15 before retrospective scoring.

**Evaluation period:**

- Start: `2019-06-01 00:00:00 UTC` — inclusive
- End: `2019-09-01 00:00:00 UTC` — exclusive
- Resolution: 5 minutes
- Evaluation observations: **26,496 per horizon**
- Alignment: Forecast **target-valid time**

**Performance metrics:**

- Mean Absolute Error (MAE), kW
- Root Mean Squared Error (RMSE), kW

**Predetermined comparison baselines:**

1. **Issue-time PV persistence:** Forecast the future PV power using the latest PV observation available at the forecast issue time.
2. **Previous-day PV persistence:** Forecast using observed PV power at the same target-valid clock time exactly 24 hours earlier.

All models and baselines must be evaluated against the same 26,496 target-valid timestamps per horizon.

Missing observations must not be silently removed.

**Prohibited operations:**

- Model refitting or retraining
- Hyperparameter optimization
- Validation-based early stopping
- Prediction clipping introduced after selection
- Feature-family or architecture changes
- Terminal-score-based model switching
- Modifying frozen model weights or training configurations

## Execution and Reproducibility

The retrospective evaluation was executed successfully using the three previously fitted and independently verified LightGBM models.

The completed execution verified:

- Frozen model SHA-256 hashes and provenance
- Feature-schema identity and native feature ordering
- Forecast issue-time and target-valid-time alignment
- Equal scoring populations across models and baselines
- Complete evaluation populations of 26,496 observations per horizon
- Finite predictions and calculated MAE/RMSE metrics
- No model fitting or hyperparameter changes

**Terminal measurements and target values were intentionally accessed for retrospective scoring.** This is the principal difference from the read-only Notebook 15.

Persistent result export remained disabled during the reviewed execution. Results were retained in the executed notebook outputs.

**Execution provenance:** The notebook was executed while the repository HEAD was `02e7526`, before the Notebook 16 scoring implementation itself was committed. Therefore, the scoring implementation must not be described as having been committed before the retrospective evaluation. Its evaluation rules were, however, documented in the earlier committed Notebook 15 protocol.

## Next Research Stage

After reviewing and archiving the SOLETE retrospective results, the project will proceed to **PVOD external validation** under a separate protocol.

PVOD validation will require explicit consideration of:

- Station-specific temporal splits and refitting
- Differences in available predictor features
- Cross-dataset feature compatibility
- Chronological data integrity
- Evaluation and baseline consistency
- The distinction between SOLETE retrospective results and PVOD external evidence

No PVOD observations are accessed in this notebook.

**Final classification:** Completed SOLETE historically exposed retrospective evaluation, with frozen models, predetermined baselines, verified alignment, and no post-evaluation tuning.

## 1. Single-purpose authorization and repository discovery

This notebook has an explicit, limited SOLETE-retrospective authorization. The frozen development-selection JSON correctly still says `terminal_access_authorized=false` as of the earlier selection stage. **Do not edit that frozen record**; the later, separate authorization for retrospective scoring applies only within this notebook. It does not authorize training or a claim of independent confirmation.

In [1]:
from __future__ import annotations
import hashlib
import importlib.metadata as metadata
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Explicit SOLETE-only authorization made after read-only Notebook 15.
RUN_SOLETE_RETROSPECTIVE = True
SOLETE_RETROSPECTIVE_TOKEN = "AUTHORIZE_HISTORICALLY_EXPOSED_SOLETE_RETROSPECTIVE"
SAVE_RETROSPECTIVE_ARTIFACTS = False  # First run: show metrics, do not write files.
TRAINING_AUTHORIZED = False
PVOD_EVALUATION_AUTHORIZED = False
assert RUN_SOLETE_RETROSPECTIVE
assert SOLETE_RETROSPECTIVE_TOKEN == "AUTHORIZE_HISTORICALLY_EXPOSED_SOLETE_RETROSPECTIVE"
assert not TRAINING_AUTHORIZED and not PVOD_EVALUATION_AUTHORIZED
assert SAVE_RETROSPECTIVE_ARTIFACTS is False  # Export requires a separately reviewed run.

SELECTION_LOCK_COMMIT = "c6a9162"
MODEL_INVENTORY_COMMIT = "565d431"
EVALUATION_PREFLIGHT_COMMIT = "02e7526"
LOCK_REL = "results/development_only/revised_model_selection_lock_v1.json"
MODEL_DIR_REL = "results/final_model_refits/revised_lightgbm_v1"
NB15_REL = "notebooks/15_revised_evaluation_protocol_preflight.ipynb"
MODEL_VERIFIER_REL = "scripts/verify_all_three_final_refits.py"

possible = ([Path(os.environ["SOLETE_PROJECT_ROOT"]).expanduser()]
            if os.environ.get("SOLETE_PROJECT_ROOT") else [])
possible += [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p.resolve() for p in possible
                     if (p / ".git").exists() and (p / LOCK_REL).is_file()
                     and (p / NB15_REL).is_file() and (p / MODEL_VERIFIER_REL).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open in the SOLETE Git checkout or set SOLETE_PROJECT_ROOT")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def git(*args, check=True):
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT,
                          capture_output=True, check=check)

for frozen in (SELECTION_LOCK_COMMIT, MODEL_INVENTORY_COMMIT, EVALUATION_PREFLIGHT_COMMIT):
    assert git("merge-base", "--is-ancestor", frozen, "HEAD", check=False).returncode == 0
print("HISTORICALLY EXPOSED SOLETE RETROSPECTIVE: EXPLICITLY AUTHORIZED")
print("TRAINING: FORBIDDEN | PVOD: NOT STARTED | OUTPUT FILE WRITES: DISABLED")
print("Checkout:", git("rev-parse", "--short", "HEAD").stdout.decode().strip())

HISTORICALLY EXPOSED SOLETE RETROSPECTIVE: EXPLICITLY AUTHORIZED
TRAINING: FORBIDDEN | PVOD: NOT STARTED | OUTPUT FILE WRITES: DISABLED
Checkout: 02e7526


## 2. Verify the frozen evaluation plan, selection lock and actual model files

Read-only verification must finish **before** opening the HDF5 file. Check that local Notebook 15 has exactly its committed bytes, the selection lock still matches commit `c6a9162`, and the independent model verifier successfully checks native `.txt` model bytes against committed JSON. The three JSON model records were frozen in commit `565d431`. Fitted model bytes remain local (outside Git).

In [2]:
assert (PROJECT_ROOT / NB15_REL).read_bytes() == git(
    "show", f"{EVALUATION_PREFLIGHT_COMMIT}:{NB15_REL}").stdout, (
    "Notebook 15 evaluation design differs from the committed protocol")
lock_bytes = (PROJECT_ROOT / LOCK_REL).read_bytes()
assert lock_bytes == git("show", f"{SELECTION_LOCK_COMMIT}:{LOCK_REL}").stdout
lock = json.loads(lock_bytes)
assert lock["archived_audit"]["archived_fold_scores"] == 1665
assert lock["evaluation_policy"]["can_claim_fresh_independent_test"] is False
assert lock["evaluation_policy"]["terminal_access_authorized"] is False
assert lock["refit_protocol"]["do_not_fit_or_score_terminal_data"] is True
# These statements describe the earlier selection stage and remain immutable.
# A *later*, explicitly authorized retrospective is separately recorded here.

specs = {int(x["horizon_minutes"]): x for x in lock["selected_candidates"]}
assert set(specs) == {15, 30, 60}
expected_family = {15: "F1", 30: "F2", 60: "F2"}
expected_features = {15: 133, 30: 138, 60: 138}
expected_rows = {15: 104817, 30: 104814, 60: 104808}
model_meta = {}
for h in (15, 30, 60):
    item = specs[h]
    assert item["model"] == "LightGBM" and item["candidate_id"] == "LightGBM_C1"
    assert item["feature_family"] == expected_family[h]
    assert item["configuration"] == {"learning_rate": 0.03, "n_estimators": 300, "num_leaves": 15}
    assert item["refit"]["training_rows_expected"] == expected_rows[h]
    name = f"lightgbm_c1_h{h}_{expected_family[h]}"
    rel = f"{MODEL_DIR_REL}/{name}.json"
    record_bytes = (PROJECT_ROOT / rel).read_bytes()
    assert record_bytes == git("show", f"{MODEL_INVENTORY_COMMIT}:{rel}").stdout
    record = json.loads(record_bytes)
    ident = record["training_identity"]
    assert ident["feature_columns"] == expected_features[h]
    assert ident["training_rows"] == expected_rows[h]
    assert ident["feature_family"] == expected_family[h]
    assert ident["lock_commit"] == SELECTION_LOCK_COMMIT
    assert record["terminal_evaluated"] is False and record["predictions_generated"] is False
    assert record["fitted_boosting_iterations"] == 300
    model_meta[h] = record

for pkg, version in lock["reproducibility"]["dependencies_pinned_from_pyproject_toml"].items():
    assert metadata.version(pkg) == version, f"{pkg} differs from frozen refit package version"

verification = subprocess.run(
    [sys.executable, MODEL_VERIFIER_REL], cwd=PROJECT_ROOT,
    capture_output=True, text=True, check=True)
assert "ALL THREE FINAL LIGHTGBM REFITS: VERIFIED" in verification.stdout
assert "NO HDF5 READS | NO TRAINING | NO PREDICTIONS | NO SCORING" in verification.stdout
print(verification.stdout)
print("SCORING PLAN, SOURCE LOCK, 3 ACTUAL MODEL FILES: VERIFIED")
print("No HDF5 measurements read yet")

15 min: VERIFIED — F1 — 104,817 development rows — 133 features — 300 trees — sha256=aeadd57863612e732c6b98972c2be80b60037dec0769331aab6e1e4c92d17bdf
30 min: VERIFIED — F2 — 104,814 development rows — 138 features — 300 trees — sha256=e841a2aa4c2ef5a19ba3c8c9ec708eea5e878c1d2c29ea625234d259eecf6eef
60 min: VERIFIED — F2 — 104,808 development rows — 138 features — 300 trees — sha256=aac4d7e3c1c84495f494f783c930ce0917d35e2441356220afffb4eb68c0f023
ALL THREE FINAL LIGHTGBM REFITS: VERIFIED — source, lock, model bytes, metadata, native structure
NO HDF5 READS | NO TRAINING | NO PREDICTIONS | NO SCORING | NO FILE WRITES
NOTE: The original refit notebooks recomputed each development-input SHA. This stand-alone audit verifies the recorded hashes' consistency without rereading SOLETE.

SCORING PLAN, SOURCE LOCK, 3 ACTUAL MODEL FILES: VERIFIED
No HDF5 measurements read yet


## 3. Lock the evaluation geometry and baselines (no HDF5 yet)

Reproduce the committed Notebook 15 reference grid entirely from timestamps before accessing measurements. The last-60-minute development-label embargo remains part of the **training**, not of the evaluation grid. Test rows are keyed by their **valid timestamps**, even when the first forecast issue is on May 31.

In [3]:
from src.forecasting_protocol import (
    FORECAST_HORIZONS, TEST_START, TEST_END, TRAIN_START,
    MODEL_AVAILABILITY_GAP, TARGET_COLUMN, F0_HISTORY_SOURCES, DAILY_LAG_STEPS,
)
assert tuple(FORECAST_HORIZONS) == (15, 30, 60)
assert MODEL_AVAILABILITY_GAP == pd.Timedelta(minutes=60)
assert (TEST_START, TEST_END) == (
    pd.Timestamp("2019-06-01 00:00:00", tz="UTC"),
    pd.Timestamp("2019-09-01 00:00:00", tz="UTC"))
assert DAILY_LAG_STEPS == 288
valid_grid = pd.date_range(TEST_START, TEST_END, freq="5min", inclusive="left", tz="UTC")
assert len(valid_grid) == 26496
assert valid_grid[0] == TEST_START
assert valid_grid[-1] == TEST_END - pd.Timedelta(minutes=5)
issue_grids = {}
for h in (15, 30, 60):
    issues = valid_grid - pd.Timedelta(minutes=h)
    assert issues[0] > pd.Timestamp("2019-05-31 22:55:00Z")
    assert (issues < valid_grid).all()
    assert ((valid_grid - pd.Timedelta(days=1)) <= issues).all()
    issue_grids[h] = issues
print("FROZEN RETROSPECTIVE DESIGN: 26,496 VALID-TIME ROWS PER HORIZON")
print("Metrics: MAE_kW / RMSE_kW")
print("Baselines: issue-time PV and valid-time-minus-24h PV")
print("NO RETRAINING / NO NEW MODEL SELECTION / NO CLIPPING / NO ROW DROPS")

FROZEN RETROSPECTIVE DESIGN: 26,496 VALID-TIME ROWS PER HORIZON
Metrics: MAE_kW / RMSE_kW
Baselines: issue-time PV and valid-time-minus-24h PV
NO RETRAINING / NO NEW MODEL SELECTION / NO CLIPPING / NO ROW DROPS


## 4. Open precisely the SOLETE measurements needed for the retrospective

This is the **first** notebook cell that reads historically exposed data. The raw file contains 131,617 five-minute timestamps, including one extra boundary timestamp on September 1, 2019. Load only the first 131,616 observations (`time < TEST_END`). The complete earlier history is required for issue-time lag features, but every feature is constructed causally from observations at or before issue time. Deterministic Gregorian and solar coordinates may be evaluated at the future **valid time**, as in development CV. No observed future weather or PV is used as a predictor.

All supplied values must be finite for the planned common 26,496-row grid. If not, **stop and report the issue**—do not drop or impute test rows without a separately locked policy.

In [4]:
import h5py
from src.preprocessing import build_cleaned_solete_features, build_gregorian_features
from src.forecasting_protocol import build_f0_history
from src.astronomy_features import (
    SOLETE_ASTRONOMY_OFFSET, SOLETE_SOLAR_ALTITUDE_CONVENTION_M,
    build_solar_f2_features,
)

assert SOLETE_ASTRONOMY_OFFSET == pd.Timedelta(minutes=2)
assert SOLETE_SOLAR_ALTITUDE_CONVENTION_M == 0.0
h5_path = PROJECT_ROOT / "solete_dataset/SOLETE_Pombo_5min.h5"
if not h5_path.is_file():
    raise FileNotFoundError(f"Local SOLETE five-minute HDF5 is missing: {h5_path}")
with h5py.File(h5_path, "r") as h5:
    data = h5["DATA"]
    full_times = pd.DatetimeIndex(
        pd.to_datetime(data["axis1"][:], unit="ns", utc=True), name="Timestamp")
    raw_columns = tuple(
        x.decode("utf-8") if isinstance(x, bytes) else str(x)
        for x in data["axis0"][:]
    )
    assert data["block0_values"].shape == (len(full_times), len(raw_columns))
    assert len(full_times) == 131617
    assert full_times[0] == TRAIN_START
    assert full_times[-1] == TEST_END
    assert full_times.is_unique and full_times.is_monotonic_increasing
    assert (full_times[1:] - full_times[:-1] == pd.Timedelta(minutes=5)).all()
    study_end = int(full_times.searchsorted(TEST_END, side="left"))
    assert study_end == 131616
    # Exclude the 2019-09-01 boundary row; don't read the unneeded trailing cell.
    raw_values = data["block0_values"][:study_end, :]

required_raw = (
    set(F0_HISTORY_SOURCES.values())
    - {"Humidity_clean", "Pressure_clean[mbar]", "wind_direction_sin", "wind_direction_cos"}
) | {TARGET_COLUMN, "Pressure[mbar]", "HUMIDITY[%]", "WIND_DIR[deg]"}
assert required_raw.issubset(raw_columns), (
    f"Missing required SOLETE raw columns: {sorted(required_raw-set(raw_columns))}")
raw = pd.DataFrame(raw_values, index=full_times[:study_end], columns=raw_columns)
del raw_values
assert raw.index[0] == TRAIN_START and raw.index[-1] == valid_grid[-1]
assert valid_grid.isin(raw.index).all()
assert all(issues.isin(raw.index).all() for issues in issue_grids.values())
assert (valid_grid - pd.Timedelta(days=1)).isin(raw.index).all()
clean = build_cleaned_solete_features(raw)
f0_full = build_f0_history(clean)
assert len(f0_full.columns) == 118
assert not f0_full.columns.duplicated().any()
print("SOLETe HISTORICAL MEASUREMENTS LOADED: 131,616 rows (through 2019-08-31 23:55 UTC)")
print("Observed future meteorology is not used as a forecasting predictor")
print("No estimator fits or learned preprocessing were executed")

SOLETe HISTORICAL MEASUREMENTS LOADED: 131,616 rows (through 2019-08-31 23:55 UTC)
Observed future meteorology is not used as a forecasting predictor
No estimator fits or learned preprocessing were executed


## 5. Fixed-model scoring with exact feature schemas

For each horizon, rebuild the **same ordered feature columns** used by its development refit: 118 causal F0 historical features (including the 24-hour PV lag alias), then 15 Gregorian valid-time indicators and, for F2, five solar valid-time variables. Compare the full ordered feature-schema SHA-256 to the *committed* training provenance and to the **native LightGBM feature names** before calling `Booster.predict`.

Two baselines are evaluated on **the identical targets**: `PV(issue_time)` and `PV(valid_time − 24 hours)`. Check the time-availability constraints explicitly. Use all **26,496** scored valid timestamps per horizon. No learning, clipping, or post-hoc filtering.

In [5]:
import lightgbm as lgb

# Explicit metrics, fixed before any retrospective outcome was inspected in this stage.
def mae_rmse_kw(y_actual, y_pred):
    actual = np.asarray(y_actual, dtype=np.float64)
    predicted = np.asarray(y_pred, dtype=np.float64)
    assert actual.shape == predicted.shape == (26496,)
    assert np.isfinite(actual).all() and np.isfinite(predicted).all()
    diff = actual - predicted
    return float(np.mean(np.abs(diff))), float(np.sqrt(np.mean(np.square(diff))))

raw_target = raw[TARGET_COLUMN]
retrospective_frames = {}
metric_records = []
schema_review = []
for h in (15, 30, 60):
    s = specs[h]
    issues, valid = issue_grids[h], valid_grid
    assert (issues + pd.Timedelta(minutes=h) == valid).all()

    # Causal observational F0: all lags are measured at/before the forecast issue.
    f0 = f0_full.loc[issues].rename(columns={
        "P_Solar_kw_lag_288": "P_Solar_kw_issue_minus_24h"
    })
    f1 = build_gregorian_features(valid).set_axis(issues).add_prefix("valid_")
    X = pd.concat([f0, f1], axis=1)
    if s["feature_family"] == "F2":
        f2 = build_solar_f2_features(valid).set_axis(issues).add_prefix("valid_")
        assert f2.shape[1] == 5
        X = pd.concat([X, f2], axis=1)
        del f2
    else:
        assert s["feature_family"] == "F1"
    assert X.index.equals(issues) and X.columns.is_unique
    assert X.shape == (26496, expected_features[h])
    assert np.isfinite(X.to_numpy(dtype=np.float64)).all(), f"h{h}: missing/invalid test features"

    # Must match the schema from the actual archived final refit, not merely the size.
    schema_digest = hashlib.sha256(
        json.dumps(list(X.columns), ensure_ascii=False).encode("utf-8")
    ).hexdigest()
    assert schema_digest == model_meta[h]["training_identity"]["feature_schema_sha256"], (
        f"h{h}: feature order or construction differs from frozen training")
    model_path = PROJECT_ROOT / MODEL_DIR_REL / model_meta[h]["model_filename"]
    booster = lgb.Booster(model_file=str(model_path))  # Load fixed model; never fit.
    assert booster.current_iteration() == 300
    assert booster.num_feature() == expected_features[h]
    assert booster.feature_name() == list(X.columns), f"h{h}: native feature order differs"

    y = raw_target.loc[valid].to_numpy(dtype=np.float64)
    persistence_issue = raw_target.loc[issues].to_numpy(dtype=np.float64)
    persistence_yesterday = raw_target.loc[valid - pd.Timedelta(days=1)].to_numpy(dtype=np.float64)
    assert len(y) == len(persistence_issue) == len(persistence_yesterday) == 26496
    assert np.isfinite(y).all(), f"h{h}: missing/invalid retrospective target"
    assert np.isfinite(persistence_issue).all(), f"h{h}: missing/invalid issue baseline"
    assert np.isfinite(persistence_yesterday).all(), f"h{h}: missing/invalid day-ahead baseline"
    assert (valid - pd.Timedelta(days=1) <= issues).all()

    forecast = np.asarray(booster.predict(X, num_iteration=300), dtype=np.float64)
    assert forecast.shape == (26496,) and np.isfinite(forecast).all()
    # No clipping of negative/above-capacity predictions, no second model choice.
    frame = pd.DataFrame({
        "horizon_minutes": h,
        "issue_time_utc": issues,
        "valid_time_utc": valid,
        "observed_pv_kw": y,
        "lightgbm_pred_kw": forecast,
        "pv_issue_persistence_kw": persistence_issue,
        "pv_valid_minus_24h_persistence_kw": persistence_yesterday,
    })
    assert len(frame) == 26496 and frame["valid_time_utc"].is_unique
    retrospective_frames[h] = frame
    schema_review.append({"horizon_minutes": h, "feature_family": s["feature_family"],
                          "feature_columns": X.shape[1], "feature_schema_sha256": schema_digest})
    for label, candidate in (
        ("LightGBM_C1_frozen", forecast),
        ("PV_at_forecast_issue", persistence_issue),
        ("PV_at_valid_time_minus_24h", persistence_yesterday),
    ):
        this_mae, this_rmse = mae_rmse_kw(y, candidate)
        metric_records.append({
            "horizon_minutes": h,
            "method": label,
            "scored_rows": len(y),
            "MAE_kW": this_mae,
            "RMSE_kW": this_rmse,
            "classification": "HISTORICALLY_EXPOSED_RETROSPECTIVE",
        })
    print(f"h{h}: 26,496 fixed-model predictions + 2 paired persistence baselines scored")
    del X, f0, f1, booster, forecast

metrics_table = pd.DataFrame(metric_records)
assert len(metrics_table) == 9
assert metrics_table["scored_rows"].eq(26496).all()
assert metrics_table.groupby("horizon_minutes")["method"].nunique().eq(3).all()
assert metrics_table[["MAE_kW", "RMSE_kW"]].ge(0).all().all()
assert (metrics_table["RMSE_kW"] + 1e-12 >= metrics_table["MAE_kW"]).all()
assert len(retrospective_frames) == 3
print("\nSOLETE RETROSPECTIVE PERFORMANCE — HISTORICALLY EXPOSED, NOT INDEPENDENT")
display(metrics_table.round(6))
display(pd.DataFrame(schema_review))

h15: 26,496 fixed-model predictions + 2 paired persistence baselines scored
h30: 26,496 fixed-model predictions + 2 paired persistence baselines scored
h60: 26,496 fixed-model predictions + 2 paired persistence baselines scored

SOLETE RETROSPECTIVE PERFORMANCE — HISTORICALLY EXPOSED, NOT INDEPENDENT


,horizon_minutes,method,scored_rows,MAE_kW,RMSE_kW,classification
0,15,LightGBM_C1_frozen,26496,0.259928,0.561341,HISTORICALLY_EXPOSED_RETROSPECTIVE
1,15,PV_at_forecast_issue,26496,0.291366,0.653352,HISTORICALLY_EXPOSED_RETROSPECTIVE
2,15,PV_at_valid_time_minus_24h,26496,0.579710,1.171413,HISTORICALLY_EXPOSED_RETROSPECTIVE
3,30,LightGBM_C1_frozen,26496,0.301437,0.622601,HISTORICALLY_EXPOSED_RETROSPECTIVE
4,30,PV_at_forecast_issue,26496,0.380360,0.762045,HISTORICALLY_EXPOSED_RETROSPECTIVE
5,30,PV_at_valid_time_minus_24h,26496,0.579710,1.171413,HISTORICALLY_EXPOSED_RETROSPECTIVE
6,60,LightGBM_C1_frozen,26496,0.341884,0.681973,HISTORICALLY_EXPOSED_RETROSPECTIVE
7,60,PV_at_forecast_issue,26496,0.520977,0.921371,HISTORICALLY_EXPOSED_RETROSPECTIVE
8,60,PV_at_valid_time_minus_24h,26496,0.579710,1.171413,HISTORICALLY_EXPOSED_RETROSPECTIVE


,horizon_minutes,feature_family,feature_columns,feature_schema_sha256
0,15,F1,133,abacbd22f6a80e771a56ee5a0c18d74e860a5775c990e4...
1,30,F2,138,6cfaaebb47ace957a7d016d8e6fd733b391bda2b0ccf82...
2,60,F2,138,6cfaaebb47ace957a7d016d8e6fd733b391bda2b0ccf82...


## 6. Descriptive paired comparisons — **no new selection**

Report the primary model alongside both predeclared baselines on each *identical* valid-time grid. Percentage change in mean absolute error is an arithmetic descriptor, **not** a p-value, probability of improvement, causal attribution, or basis for choosing another model. Do **not** compare these retrospective figures numerically to development CV means as if they were exchangeable estimands.

In [6]:
wide_mae = metrics_table.pivot(index="horizon_minutes", columns="method", values="MAE_kW")
wide_rmse = metrics_table.pivot(index="horizon_minutes", columns="method", values="RMSE_kW")
comparison = pd.DataFrame({
    "horizon_minutes": [15, 30, 60],
    "LightGBM_MAE_kW": wide_mae.loc[[15, 30, 60], "LightGBM_C1_frozen"].to_numpy(),
    "issue_persistence_MAE_kW": wide_mae.loc[[15, 30, 60], "PV_at_forecast_issue"].to_numpy(),
    "yesterday_persistence_MAE_kW": wide_mae.loc[[15, 30, 60], "PV_at_valid_time_minus_24h"].to_numpy(),
    "LightGBM_RMSE_kW": wide_rmse.loc[[15, 30, 60], "LightGBM_C1_frozen"].to_numpy(),
})
for baseline in ("issue", "yesterday"):
    col = f"{baseline}_persistence_MAE_kW"
    assert (comparison[col] > 0).all(), f"Cannot calculate descriptive MAE change: {baseline} is zero"
    comparison[f"MAE_reduction_vs_{baseline}_pct"] = (
        100.0 * (1.0 - comparison["LightGBM_MAE_kW"] / comparison[col]))
assert len(comparison) == 3
print("PAIRED DESCRIPTIVE COMPARISONS (NOT A NEW SELECTION RULE)")
display(comparison.round(5))
print("NOTE: Negative percentage means LightGBM has higher MAE than that baseline.")

PAIRED DESCRIPTIVE COMPARISONS (NOT A NEW SELECTION RULE)


,horizon_minutes,LightGBM_MAE_kW,issue_persistence_MAE_kW,yesterday_persistence_MAE_kW,LightGBM_RMSE_kW,MAE_reduction_vs_issue_pct,MAE_reduction_vs_yesterday_pct
0,15,0.25993,0.29137,0.57971,0.56134,10.79007,55.16242
1,30,0.30144,0.38036,0.57971,0.62260,20.74962,48.00211
2,60,0.34188,0.52098,0.57971,0.68197,34.37627,41.02493


NOTE: Negative percentage means LightGBM has higher MAE than that baseline.


## 7. Final execution integrity and reporting boundary

All three fixed refits should have produced exactly **79,488 model forecasts** in total, plus two equal-length baseline series each. No modified weights, new fits, hidden data exclusions, model swapping, or new learned preprocessing are permitted. In this initial review run results exist in notebook memory/output only; **they are not automatically written to results/**.

**After the executed notebook is audited:** archive scoring tables and, if desired, all timestamp-level predictions in a separately checked results bundle with model/input hashes and explicit `historically_exposed=true`. Keep it outside the development-CV archive, and do not overwrite original historical notebooks/results. Only then design the **separate** PVOD evaluation, where station differences and transfer assumptions require their own frozen protocol.

In [7]:
assert RUN_SOLETE_RETROSPECTIVE and not TRAINING_AUTHORIZED
assert not PVOD_EVALUATION_AUTHORIZED and not SAVE_RETROSPECTIVE_ARTIFACTS
assert set(retrospective_frames) == {15, 30, 60}
assert sum(len(x) for x in retrospective_frames.values()) == 3 * 26496
assert len(metrics_table) == 9
assert len(comparison) == 3
for h, frame in retrospective_frames.items():
    assert frame["horizon_minutes"].eq(h).all()
    assert pd.DatetimeIndex(frame["valid_time_utc"]).equals(valid_grid)
    assert (frame["issue_time_utc"] + pd.Timedelta(minutes=h)
            == frame["valid_time_utc"]).all()
    assert frame[["observed_pv_kw", "lightgbm_pred_kw",
                  "pv_issue_persistence_kw", "pv_valid_minus_24h_persistence_kw"]].notna().all().all()
print("NOTEBOOK 16 SOLETE RETROSPECTIVE SCORING: COMPLETED — HISTORICALLY EXPOSED")
print("MODEL INVENTORY: 3/3 frozen LightGBM files verified before scoring")
print("MODEL PREDICTIONS: 79,488 | PAIRED BASELINES: 158,976 | TARGET VALID-TIMES: 26,496/horizon")
print("MAE/RMSE: reported on all rows; no silent row drops, fitting, tuning or winner switching")
print("EXPORT: DISABLED — notebook results available for review; no result files written")
print("INDEPENDENT TEST CLAIM: FORBIDDEN | NEXT STAGE: PVOD EXTERNAL EVALUATION (NOT RUN)")

NOTEBOOK 16 SOLETE RETROSPECTIVE SCORING: COMPLETED — HISTORICALLY EXPOSED
MODEL INVENTORY: 3/3 frozen LightGBM files verified before scoring
MODEL PREDICTIONS: 79,488 | PAIRED BASELINES: 158,976 | TARGET VALID-TIMES: 26,496/horizon
MAE/RMSE: reported on all rows; no silent row drops, fitting, tuning or winner switching
EXPORT: DISABLED — notebook results available for review; no result files written
INDEPENDENT TEST CLAIM: FORBIDDEN | NEXT STAGE: PVOD EXTERNAL EVALUATION (NOT RUN)
